# SC Experiment V2
Four focused experiments with full pipeline: tree generation, visualization,
voxel mesh synthesis, STL export, domain embedding, STL export of embedded artifact.

## Setup & Imports

In [ ]:
import sys, os, time
from pathlib import Path

ROOT = str(Path(".").resolve().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from test.space_colonization_runner import run_space_colonization, run_space_colonization_dual_tree
from test.odc_runner import run_odc
from test.notebook_utils import (
    plot_network_2d,
    plot_network_3d,
    print_stats,
    compare_networks,
    compare_stats_table,
    network_to_dataframe,
    save_network_json,
)

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
%matplotlib inline

## GPU & Performance Status
Check what acceleration backends are available for the SC algorithm.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s %(levelname)s: %(message)s")

from generation.ops._gpu_nn import gpu_available, _TORCH_AVAILABLE, _TORCH_CUDA_AVAILABLE, _GPU_MIN_QUERIES, _warmup_gpu

print('=== SC Performance Optimization Status ===')
print(f'  PyTorch CUDA NN:  {"ENABLED" if _TORCH_CUDA_AVAILABLE else "not available (using scipy cKDTree)"}')
print(f'  PyTorch dir avg:  {"ENABLED" if _TORCH_CUDA_AVAILABLE else "not available (using numpy)"}')
print(f'  Spatial hashing:  ENABLED (auto for >5k nodes)')
print(f'  GPU min queries:  {_GPU_MIN_QUERIES} (below this, CPU is used)')
print(f'  Vectorized ops:   ENABLED (boolean masking, batch kill-radius)')
print(f'  Contiguous arrays: ENABLED (pre-allocated position tracking)')
print()
if not _TORCH_CUDA_AVAILABLE:
    print('To enable GPU acceleration: pip install torch')
    print('(PyTorch will auto-detect CUDA if available)')

from generation.ops._gpu_nn import batch_collision_prefilter, batch_direction_average, PersistentGPUIndex
print(f'  Batch collision:  ENABLED (Phase 2 midpoint-distance pre-filter)')
print(f'  Batch directions: ENABLED (Phase 3 vectorized across all tips)')
print(f'  PersistentGPU:    {"ENABLED" if _TORCH_CUDA_AVAILABLE else "CPU fallback"} (Phase 4 persistent tissue on GPU)')


## Shared Helpers

In [ ]:
from generation.ops.mesh.synthesis import synthesize_mesh, MeshSynthesisPolicy
from generation.api.embed import embed_void
from aog_policies.generation import EmbeddingPolicy
from generation.core.domain import CylinderDomain
from generation.core.types import Point3D

def summarize(net, label=""):
    n = len(net.nodes)
    s = len(net.segments)
    t = sum(1 for nd in net.nodes.values() if nd.node_type == "terminal")
    print(f"{label}  nodes={n}  segments={s}  terminals={t}")

mesh_policy = {
    "add_node_spheres": False,
    "cap_ends": True,
    "segments_per_circle": 16,
    "radius_clamp_min": 1e-4,
    "radius_clamp_max": None,
    "voxel_repair_synthesis": True,
    "voxel_repair_pitch": 2e-4,
    "voxel_repair_auto_adjust": True,
    "voxel_repair_max_steps": 6,
    "voxel_repair_step_factor": 1.5,
    "voxel_repair_max_voxels": 50_000_000,
    "max_voxels": 50_000_000,
}

embedding_policy = {
    "voxel_pitch": 3e-4,
    "shell_thickness": 2e-3,
    "auto_adjust_pitch": True,
    "max_pitch_steps": 4,
    "max_voxels": 50_000_000,
    "preserve_ports_enabled": True,
}

def build_mesh(network, label="mesh"):
    mp = MeshSynthesisPolicy.from_dict(mesh_policy)
    mesh, report = synthesize_mesh(network, policy=mp)
    meta = report.metadata
    print(f"{label}: verts={meta.get('vertex_count',0)}  faces={meta.get('face_count',0)}  "
          f"watertight={meta.get('is_watertight')}  voxel_repair={meta.get('voxel_repair_applied')}")
    return mesh, report

def show_mesh_3d(mesh, title="Mesh", color="crimson"):
    v, f = mesh.vertices, mesh.faces
    fig = go.Figure(data=[go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        opacity=0.6, color=color, flatshading=True,
    )])
    fig.update_layout(title=title, scene=dict(aspectmode="data"), width=800, height=600)
    fig.show()

def export_stl(mesh, path, label=""):
    mesh.export(path)
    v, f = mesh.vertices, mesh.faces
    print(f"{label} STL -> {path}  ({len(v)} verts, {len(f)} faces, watertight={mesh.is_watertight})")

def run_embedding(domain_spec, void_mesh, ports=None, label="embedding"):
    ep = EmbeddingPolicy.from_dict(embedding_policy)
    solid, void_out, shell, report = embed_void(domain_spec, void_mesh, ep, ports=ports)
    meta = report.metadata
    print(f"{label}: solid_vol={meta.get('solid_volume','N/A')}  void_vol={meta.get('void_volume','N/A')}  "
          f"pitch={meta.get('effective_pitch','N/A')}")
    if report.warnings:
        for w in report.warnings:
            print(f"  WARNING: {w}")
    return solid, void_out, shell, report

## Tissue Sampling Configuration

In [ ]:
tissue_sampling = {
    "enabled": True,
    "n_points": 15000,
    "strategy": "uniform",
    "depth_reference": {"mode": "face", "face": "top"},
    "depth_distribution": "power",
    "depth_power": 2.0,
    "seed": 42,
}

## Base SC Parameters

In [ ]:
sc_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.1,
    "domain_height": 0.3,
    "domain_center": [0.0, 0.0, 0.0],

    "inlet_position": [0.0, 0.0, 0.15],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",

    "num_attractors": 15000,
    "attraction_distance": 0.015,
    "kill_distance": 0.003,
    "step_size": 0.0001,
    "max_iterations": 512,
    "max_steps": 512,
    "branch_angle_deg": 35.0,
    "directional_bias": 0.85,
    "max_deviation_deg": 25.0,

    "encourage_bifurcation": True,
    "max_children_per_node": 2,
    "bifurcation_probability": 0.8,
    "min_attractions_for_bifurcation": 3,
    "bifurcation_angle_threshold_deg": 55.0,

    "min_radius": 0.0001,
    "taper_factor": 0.95,

    "progress": True,
    "kdtree_rebuild_tip_every": 5,
    "kdtree_rebuild_all_nodes_every": 15,
    "stall_steps_per_inlet": 10,
    "interleaving_strategy": "round_robin",

    "check_collisions": True,
    "collision_clearance": 0.0002,
    "collision_merge_distance": 0.0003,

    "seed": 42,
    "num_outlets": 50,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": tissue_sampling,
}


---
# Experiment 0 -- GPU vs CPU Speed Comparison (Multi-Scale)
Runs SC at increasing attractor counts (2k, 5k, 10k, 15k) on a small volume
with 100 iterations each. Compares wall-clock time with GPU acceleration
enabled vs forced CPU fallback to find the crossover point.

### 0a. GPU Warmup

In [ ]:
import generation.ops._gpu_nn as _gpu_mod

print(f"GPU available: {_gpu_mod.gpu_available()}")
print(f"PyTorch CUDA:  {_gpu_mod._TORCH_CUDA_AVAILABLE}")
print(f"GPU min queries threshold: {_gpu_mod._GPU_MIN_QUERIES}")
print()

if _gpu_mod._TORCH_CUDA_AVAILABLE:
    _gpu_mod._warmup_gpu()
    print("GPU warmup complete (JIT overhead eliminated)")
else:
    print("No CUDA GPU -- both runs will use CPU (scipy cKDTree)")

### 0b. Multi-Scale Benchmark

In [ ]:
exp0_scales = [2000, 5000, 10000, 15000]
exp0_results = []

for n_attr in exp0_scales:
    exp0_tissue = {
        "enabled": True,
        "n_points": n_attr,
        "strategy": "uniform",
        "seed": 42,
    }

    exp0_params = {
        "domain_type": "cylinder",
        "domain_radius": 0.02,
        "domain_height": 0.06,
        "domain_center": [0.0, 0.0, 0.0],
        "inlet_position": [0.0, 0.0, 0.03],
        "inlet_radius": 0.001,
        "vessel_type": "arterial",
        "num_attractors": n_attr,
        "attraction_distance": 0.008,
        "kill_distance": 0.002,
        "step_size": 0.0001,
        "max_iterations": 100,
        "max_steps": 100,
        "branch_angle_deg": 35.0,
        "directional_bias": 0.85,
        "max_deviation_deg": 25.0,
        "encourage_bifurcation": True,
        "max_children_per_node": 2,
        "bifurcation_probability": 0.8,
        "min_attractions_for_bifurcation": 3,
        "bifurcation_angle_threshold_deg": 55.0,
        "min_radius": 0.0001,
        "taper_factor": 0.95,
        "progress": False,
        "kdtree_rebuild_tip_every": 5,
        "kdtree_rebuild_all_nodes_every": 15,
        "stall_steps_per_inlet": 10,
        "interleaving_strategy": "round_robin",
        "check_collisions": True,
        "collision_clearance": 0.0002,
        "collision_merge_distance": 0.0003,
        "seed": 42,
        "num_outlets": 10,
        "apply_murray": True,
        "murray_exponent": 3.0,
        "terminal_radius": 0.0003,
        "tissue_sampling": exp0_tissue,
    }

    print(f"\n--- {n_attr} attractors ---")

    _gpu_mod.invalidate_gpu_cache()
    t0 = time.perf_counter()
    net_gpu, stats_gpu = run_space_colonization(exp0_params)
    elapsed_gpu = time.perf_counter() - t0
    print(f"  GPU/optimized: {elapsed_gpu:.3f}s  ({stats_gpu.get('nodes', '?')} nodes)")

    saved = _gpu_mod._TORCH_CUDA_AVAILABLE
    _gpu_mod._TORCH_CUDA_AVAILABLE = False
    try:
        t0 = time.perf_counter()
        net_cpu, stats_cpu = run_space_colonization(exp0_params)
        elapsed_cpu = time.perf_counter() - t0
        print(f"  CPU fallback:  {elapsed_cpu:.3f}s  ({stats_cpu.get('nodes', '?')} nodes)")
    finally:
        _gpu_mod._TORCH_CUDA_AVAILABLE = saved

    speedup = elapsed_cpu / elapsed_gpu if elapsed_gpu > 0 else 0
    print(f"  Speedup: {speedup:.2f}x {'(GPU faster)' if speedup > 1 else '(CPU faster)'}")

    exp0_results.append({
        "attractors": n_attr,
        "gpu_time": elapsed_gpu,
        "cpu_time": elapsed_cpu,
        "gpu_nodes": stats_gpu.get("nodes", 0),
        "cpu_nodes": stats_cpu.get("nodes", 0),
        "speedup": speedup,
    })

### 0c. Results Summary & Crossover Chart

In [ ]:
print("=" * 70)
print("  Experiment 0: GPU vs CPU Speed Comparison (Multi-Scale)")
print("=" * 70)
print(f"  Domain: cylinder r=0.02 h=0.06 | Max steps: 100")
print(f"  GPU available: {_gpu_mod._TORCH_CUDA_AVAILABLE}")
print(f"  GPU min queries threshold: {_gpu_mod._GPU_MIN_QUERIES}")
print()
print(f"  {'Attractors':>10}  {'GPU (s)':>10}  {'CPU (s)':>10}  {'Speedup':>10}  {'Winner':>10}")
print(f"  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")
for r in exp0_results:
    winner = 'GPU' if r['speedup'] > 1 else 'CPU'
    print(f"  {r['attractors']:>10,}  {r['gpu_time']:>10.3f}  {r['cpu_time']:>10.3f}  {r['speedup']:>9.2f}x  {winner:>10}")
print("=" * 70)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

scales = [r['attractors'] for r in exp0_results]
gpu_times = [r['gpu_time'] for r in exp0_results]
cpu_times = [r['cpu_time'] for r in exp0_results]
speedups = [r['speedup'] for r in exp0_results]

ax1.plot(scales, gpu_times, 'o-', label='GPU/optimized', color='green')
ax1.plot(scales, cpu_times, 's-', label='CPU fallback', color='blue')
ax1.set_xlabel('Number of Attractors')
ax1.set_ylabel('Wall-clock Time (s)')
ax1.set_title('Execution Time: GPU vs CPU')
ax1.legend()
ax1.grid(True, alpha=0.3)

colors = ['green' if s > 1 else 'red' for s in speedups]
ax2.bar(range(len(scales)), speedups, color=colors, tick_label=[f'{s//1000}k' for s in scales])
ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='breakeven')
ax2.set_xlabel('Number of Attractors')
ax2.set_ylabel('Speedup (GPU / CPU)')
ax2.set_title('GPU Speedup Factor')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
# Experiment 0b -- GPU vs CPU by Step Count
Fixed 10k attractors, varying max steps (10, 50, 100, 200) to see how
iteration count affects the GPU vs CPU tradeoff.

### 0b-a. Step Count Benchmark

In [ ]:
exp0b_steps = [10, 50, 100, 200]
exp0b_results = []

exp0b_tissue = {
    "enabled": True,
    "n_points": 10000,
    "strategy": "uniform",
    "seed": 42,
}

exp0b_base = {
    "domain_type": "cylinder",
    "domain_radius": 0.02,
    "domain_height": 0.06,
    "domain_center": [0.0, 0.0, 0.0],
    "inlet_position": [0.0, 0.0, 0.03],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",
    "num_attractors": 10000,
    "attraction_distance": 0.008,
    "kill_distance": 0.002,
    "step_size": 0.0001,
    "branch_angle_deg": 35.0,
    "directional_bias": 0.85,
    "max_deviation_deg": 25.0,
    "encourage_bifurcation": True,
    "max_children_per_node": 2,
    "bifurcation_probability": 0.8,
    "min_attractions_for_bifurcation": 3,
    "bifurcation_angle_threshold_deg": 55.0,
    "min_radius": 0.0001,
    "taper_factor": 0.95,
    "progress": False,
    "kdtree_rebuild_tip_every": 5,
    "kdtree_rebuild_all_nodes_every": 15,
    "stall_steps_per_inlet": 10,
    "interleaving_strategy": "round_robin",
    "check_collisions": True,
    "collision_clearance": 0.0002,
    "collision_merge_distance": 0.0003,
    "seed": 42,
    "num_outlets": 10,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": exp0b_tissue,
}

for n_steps in exp0b_steps:
    params = {**exp0b_base, "max_iterations": n_steps, "max_steps": n_steps}

    print(f"\n--- {n_steps} steps (10k attractors) ---")

    _gpu_mod.invalidate_gpu_cache()
    t0 = time.perf_counter()
    net_gpu, stats_gpu = run_space_colonization(params)
    elapsed_gpu = time.perf_counter() - t0
    print(f"  GPU/optimized: {elapsed_gpu:.3f}s  ({stats_gpu.get('nodes', '?')} nodes, {stats_gpu.get('segments', '?')} segments)")

    saved = _gpu_mod._TORCH_CUDA_AVAILABLE
    _gpu_mod._TORCH_CUDA_AVAILABLE = False
    try:
        t0 = time.perf_counter()
        net_cpu, stats_cpu = run_space_colonization(params)
        elapsed_cpu = time.perf_counter() - t0
        print(f"  CPU fallback:  {elapsed_cpu:.3f}s  ({stats_cpu.get('nodes', '?')} nodes, {stats_cpu.get('segments', '?')} segments)")
    finally:
        _gpu_mod._TORCH_CUDA_AVAILABLE = saved

    speedup = elapsed_cpu / elapsed_gpu if elapsed_gpu > 0 else 0
    print(f"  Speedup: {speedup:.2f}x {'(GPU faster)' if speedup > 1 else '(CPU faster)'}")

    exp0b_results.append({
        "steps": n_steps,
        "gpu_time": elapsed_gpu,
        "cpu_time": elapsed_cpu,
        "gpu_nodes": stats_gpu.get("nodes", 0),
        "cpu_nodes": stats_cpu.get("nodes", 0),
        "speedup": speedup,
    })

### 0b-b. Results Summary & Chart

In [ ]:
print("=" * 70)
print("  Experiment 0b: GPU vs CPU by Step Count (10k attractors)")
print("=" * 70)
print(f"  Domain: cylinder r=0.02 h=0.06 | Attractors: 10,000")
print(f"  GPU available: {_gpu_mod._TORCH_CUDA_AVAILABLE}")
print()
print(f"  {'Steps':>10}  {'GPU (s)':>10}  {'CPU (s)':>10}  {'Speedup':>10}  {'Nodes':>10}  {'Winner':>10}")
print(f"  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")
for r in exp0b_results:
    winner = 'GPU' if r['speedup'] > 1 else 'CPU'
    print(f"  {r['steps']:>10}  {r['gpu_time']:>10.3f}  {r['cpu_time']:>10.3f}  {r['speedup']:>9.2f}x  {r['gpu_nodes']:>10}  {winner:>10}")
print("=" * 70)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

steps_list = [r['steps'] for r in exp0b_results]
gpu_times = [r['gpu_time'] for r in exp0b_results]
cpu_times = [r['cpu_time'] for r in exp0b_results]
speedups = [r['speedup'] for r in exp0b_results]

ax1.plot(steps_list, gpu_times, 'o-', label='GPU/optimized', color='green')
ax1.plot(steps_list, cpu_times, 's-', label='CPU fallback', color='blue')
ax1.set_xlabel('Max Steps')
ax1.set_ylabel('Wall-clock Time (s)')
ax1.set_title('Execution Time vs Step Count (10k attractors)')
ax1.legend()
ax1.grid(True, alpha=0.3)

colors = ['green' if s > 1 else 'red' for s in speedups]
ax2.bar(range(len(steps_list)), speedups, color=colors, tick_label=[str(s) for s in steps_list])
ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='breakeven')
ax2.set_xlabel('Max Steps')
ax2.set_ylabel('Speedup (GPU / CPU)')
ax2.set_title('GPU Speedup vs Step Count')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
# Experiment 0c -- Phase 2-4 Optimization Benchmark
Compares SC performance with incremental optimization phases enabled:
- **Baseline**: Phase 1 only (spatial-indexed collision)
- **+Phase 2**: Batch collision pre-filter (midpoint-distance check skips narrow-phase for clearly-safe candidates)
- **+Phase 3**: Batch direction computation (vectorized across all tips per step)
- **+Phase 4**: PersistentGPUIndex (tissue points stay on GPU, no per-step CPU→GPU transfer)

Uses 10k attractors, 100 steps on a small domain for quick comparison.

In [ ]:
import time
import generation.ops._gpu_nn as _gpu_mod
from generation.ops._gpu_nn import PersistentGPUIndex, batch_collision_prefilter, batch_direction_average

exp0c_tissue = {
    "enabled": True,
    "n_points": 10000,
    "strategy": "uniform",
    "seed": 42,
}

exp0c_base = {
    "domain_type": "cylinder",
    "domain_radius": 0.02,
    "domain_height": 0.06,
    "domain_center": [0.0, 0.0, 0.0],
    "inlet_position": [0.0, 0.0, 0.03],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",
    "num_attractors": 10000,
    "attraction_distance": 0.015,
    "kill_distance": 0.003,
    "step_size": 0.0001,
    "max_iterations": 100,
    "max_steps": 100,
    "branch_angle_deg": 35.0,
    "directional_bias": 0.85,
    "max_deviation_deg": 25.0,
    "encourage_bifurcation": True,
    "max_children_per_node": 2,
    "bifurcation_probability": 0.8,
    "min_attractions_for_bifurcation": 3,
    "bifurcation_angle_threshold_deg": 55.0,
    "min_radius": 0.0001,
    "taper_factor": 0.95,
    "progress": False,
    "kdtree_rebuild_tip_every": 5,
    "kdtree_rebuild_all_nodes_every": 15,
    "stall_steps_per_inlet": 10,
    "check_collisions": True,
    "collision_clearance": 0.0002,
    "collision_merge_distance": 0.0003,
    "seed": 42,
    "num_outlets": 50,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "tissue_sampling": exp0c_tissue,
}

print("Running Phase 2-4 optimization benchmark (10k attractors, 100 steps)...")
print(f"GPU available: {_gpu_mod.gpu_available()}")
print(f"PersistentGPUIndex available: {_gpu_mod._TORCH_CUDA_AVAILABLE}")
print()

t0 = time.perf_counter()
net_0c, stats_0c = run_space_colonization(exp0c_base)
elapsed_0c = time.perf_counter() - t0

nodes_0c = len(net_0c.nodes)
segs_0c = len(net_0c.segments)

print("=" * 70)
print("  Experiment 0c: Phase 2-4 Optimization Benchmark")
print("=" * 70)
print(f"  Domain:       cylinder r=0.02 h=0.06")
print(f"  Attractors:   10,000")
print(f"  Max steps:    100")
print(f"  GPU:          {_gpu_mod._TORCH_CUDA_AVAILABLE}")
print(f"  Phases:       1 (spatial collision) + 2 (batch prefilter) + 3 (batch dirs) + 4 (persistent GPU)")
print(f"")
print(f"  All phases:   {elapsed_0c:.3f}s  ({nodes_0c} nodes, {segs_0c} segments)")
print(f"")
print(f"  Optimization breakdown:")
print(f"    Phase 1: Spatial-indexed collision (O(N) -> O(log N)) -- dominant bottleneck")
print(f"    Phase 2: Batch collision pre-filter -- skips narrow-phase for safe candidates")
print(f"    Phase 3: Batch direction average -- vectorized across all tips")
print(f"    Phase 4: PersistentGPUIndex -- tissue stays on GPU (if CUDA available)")
print("=" * 70)


---
# Experiment 1 -- Regular Space Colonization (Single Inlet)

### 1a. Generate Tree

In [ ]:
t0 = time.perf_counter()
sc_net, sc_stats = run_space_colonization(sc_params)
sc_elapsed = time.perf_counter() - t0
print_stats(sc_stats, "SC single inlet")
print(f"elapsed: {sc_elapsed:.2f}s")

### 1b. Visualize Tree

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(sc_net, projection=proj, ax=ax, title=f"SC single ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_net, title="SC Single-Inlet 3D")
fig3d.show()

### 1c. Voxel Mesh Synthesis

In [ ]:
sc_mesh, sc_mesh_report = build_mesh(sc_net, "SC single-inlet mesh")

### 1d. View & Export Void Mesh STL

In [ ]:
show_mesh_3d(sc_mesh, title="SC Single-Inlet Void Mesh", color="crimson")
export_stl(sc_mesh, "exp1_sc_single_void.stl", "Exp1 void")

### 1e. Embed into Domain

In [ ]:
domain_1 = CylinderDomain(radius=0.1, height=0.3, center=Point3D(0, 0, 0))
ports_1 = [{"position": (0.0, 0.0, 0.15), "direction": (0, 0, -1), "radius": 0.001}]
solid_1, void_1, shell_1, embed_report_1 = run_embedding(domain_1, sc_mesh, ports=ports_1, label="Exp1 embedding")

### 1f. View & Export Embedded STL

In [ ]:
show_mesh_3d(solid_1, title="Exp1 -- Embedded Solid (domain + void)", color="lightcoral")
export_stl(solid_1, "exp1_sc_single_solid.stl", "Exp1 solid")

---
# Experiment 2 -- Multi-Inlet Forest with Merge

### 2a. Generate Tree

In [ ]:
sc_forest_params = {
    **sc_params,
    "inlets": [
        {"position": [0.03, 0.0, 0.15], "radius": 0.001},
        {"position": [-0.03, 0.0, 0.15], "radius": 0.001},
        {"position": [0.0, 0.03, 0.15], "radius": 0.001},
    ],
    "multi_inlet_mode": "forest",
    "multi_inlet_blend_sigma": 0.0,
    "collision_merge_distance": 0.0003,
    "max_inlets": 10,
    "seed": 123,
}

t0 = time.perf_counter()
forest_net, forest_stats = run_space_colonization(sc_forest_params)
forest_elapsed = time.perf_counter() - t0
print_stats(forest_stats, "SC forest (multi-inlet, merge)")
print(f"elapsed: {forest_elapsed:.2f}s")

### 2b. Visualize Tree

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(forest_net, projection=proj, ax=ax, title=f"Forest ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(forest_net, title="SC Forest Multi-Inlet 3D")
fig3d.show()

### 2c. Voxel Mesh Synthesis

In [ ]:
forest_mesh, forest_mesh_report = build_mesh(forest_net, "Forest mesh")

### 2d. View & Export Void Mesh STL

In [ ]:
show_mesh_3d(forest_mesh, title="Forest Void Mesh", color="steelblue")
export_stl(forest_mesh, "exp2_forest_void.stl", "Exp2 void")

### 2e. Embed into Domain

In [ ]:
domain_2 = CylinderDomain(radius=0.1, height=0.3, center=Point3D(0, 0, 0))
ports_2 = [
    {"position": (0.03, 0.0, 0.15), "direction": (0, 0, -1), "radius": 0.001},
    {"position": (-0.03, 0.0, 0.15), "direction": (0, 0, -1), "radius": 0.001},
    {"position": (0.0, 0.03, 0.15), "direction": (0, 0, -1), "radius": 0.001},
]
solid_2, void_2, shell_2, embed_report_2 = run_embedding(domain_2, forest_mesh, ports=ports_2, label="Exp2 embedding")

### 2f. View & Export Embedded STL

In [ ]:
show_mesh_3d(solid_2, title="Exp2 -- Forest Embedded Solid", color="cornflowerblue")
export_stl(solid_2, "exp2_forest_solid.stl", "Exp2 solid")

---
# Experiment 3 -- Dual Tree (No Merge, 20 um Collision Including Radius)

### 3a. Generate Dual Tree

In [ ]:
sc_dual_params = {
    **sc_params,
    "inlets": [
        {"position": [0.0, 0.0, 0.15], "radius": 0.001},
        {"position": [0.0, 0.0, -0.15], "radius": 0.001},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "check_collisions": True,
    "collision_clearance": 20e-6,
    "collision_merge_distance": 0.0,
    "min_segment_length": 0.002,
    "max_inlets": 10,
    "seed": 42,
}

t0 = time.perf_counter()
dual_net, dual_stats = run_space_colonization(sc_dual_params)
dual_elapsed = time.perf_counter() - t0
print_stats(dual_stats, "SC dual tree (no merge, 20um collision)")
print(f"elapsed: {dual_elapsed:.2f}s")

### 3b. Visualize Dual Tree

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(dual_net, projection=proj, ax=ax, title=f"Dual tree ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(dual_net, title="SC Dual Tree 3D (no merge, 20um collision)")
fig3d.show()

### 3c. Voxel Mesh Synthesis

In [ ]:
dual_mesh, dual_mesh_report = build_mesh(dual_net, "Dual tree mesh")

### 3d. View & Export Void Mesh STL

In [ ]:
show_mesh_3d(dual_mesh, title="Dual Tree Void Mesh", color="mediumseagreen")
export_stl(dual_mesh, "exp3_dual_void.stl", "Exp3 void")

### 3e. Embed into Domain

In [ ]:
domain_3 = CylinderDomain(radius=0.1, height=0.3, center=Point3D(0, 0, 0))
ports_3 = [
    {"position": (0.0, 0.0, 0.15), "direction": (0, 0, -1), "radius": 0.001},
    {"position": (0.0, 0.0, -0.15), "direction": (0, 0, 1), "radius": 0.001},
]
solid_3, void_3, shell_3, embed_report_3 = run_embedding(domain_3, dual_mesh, ports=ports_3, label="Exp3 embedding")

### 3f. View & Export Embedded STL

In [ ]:
show_mesh_3d(solid_3, title="Exp3 -- Dual Tree Embedded Solid", color="darkseagreen")
export_stl(solid_3, "exp3_dual_solid.stl", "Exp3 solid")

---
# Experiment 4 -- Hierarchical Tissue Sampling
Exposes the tissue sampling procedure: generate priority-ordered tissue levels (L0 deep core,
L1 mid-range, L2 outer filler), visualize them, then feed to ODC with Murray's law.

### 4a. Generate Hierarchical Tissue Levels

In [ ]:
rng_hier = np.random.default_rng(42)

hier_l0 = np.column_stack([
    rng_hier.normal(0, 0.02, 200),
    rng_hier.normal(0, 0.02, 200),
    rng_hier.uniform(-0.12, -0.06, 200),
])

hier_l1 = np.column_stack([
    rng_hier.uniform(-0.06, 0.06, 400),
    rng_hier.uniform(-0.06, 0.06, 400),
    rng_hier.uniform(-0.10, 0.04, 400),
])

hier_l2 = np.column_stack([
    rng_hier.uniform(-0.09, 0.09, 600),
    rng_hier.uniform(-0.09, 0.09, 600),
    rng_hier.uniform(-0.14, 0.14, 600),
])

def clip_to_cylinder(pts, radius, half_h):
    r = np.sqrt(pts[:, 0]**2 + pts[:, 1]**2)
    mask = (r < radius) & (np.abs(pts[:, 2]) < half_h)
    return pts[mask]

hier_l0 = clip_to_cylinder(hier_l0, 0.1, 0.15)
hier_l1 = clip_to_cylinder(hier_l1, 0.1, 0.15)
hier_l2 = clip_to_cylinder(hier_l2, 0.1, 0.15)

print(f"Tissue levels -- L0 (deep core): {len(hier_l0)}")
print(f"                 L1 (mid-range): {len(hier_l1)}")
print(f"                 L2 (outer):     {len(hier_l2)}")
print(f"                 Total:          {len(hier_l0)+len(hier_l1)+len(hier_l2)}")

### 4b. Visualize Tissue Levels

In [ ]:
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121)
ax1.scatter(hier_l0[:, 0]*1e3, hier_l0[:, 2]*1e3, s=4, alpha=0.6, label="L0 deep core", c="red")
ax1.scatter(hier_l1[:, 0]*1e3, hier_l1[:, 2]*1e3, s=3, alpha=0.4, label="L1 mid-range", c="orange")
ax1.scatter(hier_l2[:, 0]*1e3, hier_l2[:, 2]*1e3, s=2, alpha=0.3, label="L2 outer", c="blue")
ax1.set_xlabel("X (mm)")
ax1.set_ylabel("Z (mm)")
ax1.set_title("Tissue Levels -- XZ projection")
ax1.legend()
ax1.set_aspect("equal")

ax2 = fig.add_subplot(122, projection="3d")
ax2.scatter(hier_l0[:, 0]*1e3, hier_l0[:, 1]*1e3, hier_l0[:, 2]*1e3, s=4, alpha=0.6, label="L0", c="red")
ax2.scatter(hier_l1[:, 0]*1e3, hier_l1[:, 1]*1e3, hier_l1[:, 2]*1e3, s=3, alpha=0.4, label="L1", c="orange")
ax2.scatter(hier_l2[:, 0]*1e3, hier_l2[:, 1]*1e3, hier_l2[:, 2]*1e3, s=2, alpha=0.3, label="L2", c="blue")
ax2.set_xlabel("X (mm)")
ax2.set_ylabel("Y (mm)")
ax2.set_zlabel("Z (mm)")
ax2.set_title("Tissue Levels -- 3D")
ax2.legend()
plt.tight_layout()
plt.show()

### 4c. Run ODC with Hierarchical Tissue (Murray Enabled)

In [ ]:
odc_hier_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.1,
    "domain_height": 0.3,
    "domain_center": [0.0, 0.0, 0.0],
    "inlet_position": [0.0, 0.0, 0.15],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",
    "tissue_levels": [
        {"priority": 0, "points": hier_l0.tolist(), "label": "deep core",
         "weight": 1.0, "coverage_threshold": 0.8},
        {"priority": 1, "points": hier_l1.tolist(), "label": "mid-range",
         "weight": 0.6, "coverage_threshold": 0.7},
        {"priority": 2, "points": hier_l2.tolist(), "label": "outer filler",
         "weight": 0.3, "coverage_threshold": 0.5},
    ],
    "augment_with_filler": False,
    "influence_radius": 0.015,
    "kill_radius": 0.003,
    "step_size": 0.0001,
    "max_steps": 512,
    "bifurcation_probability": 0.8,
    "max_children_per_node": 2,
    "taper_factor": 0.95,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "seed": 42,
}

t0 = time.perf_counter()
hier_net, hier_stats = run_odc(odc_hier_params)
hier_elapsed = time.perf_counter() - t0
summarize(hier_net, "ODC hierarchical tissue")
print(f"  elapsed={hier_elapsed:.2f}s  iters={hier_stats['iterations_used']}")
print(f"  levels_reached={hier_stats['levels_reached']}")

### 4d. Visualize Tree

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(hier_net, projection=proj, ax=ax, title=f"ODC hierarchical ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(hier_net, title="ODC Hierarchical Tissue 3D")
fig3d.show()

### 4e. Voxel Mesh Synthesis

In [ ]:
hier_mesh, hier_mesh_report = build_mesh(hier_net, "Hierarchical tissue mesh")

### 4f. View & Export Void Mesh STL

In [ ]:
show_mesh_3d(hier_mesh, title="Hierarchical Tissue Void Mesh", color="darkorchid")
export_stl(hier_mesh, "exp4_hier_void.stl", "Exp4 void")

### 4g. Embed into Domain

In [ ]:
domain_4 = CylinderDomain(radius=0.1, height=0.3, center=Point3D(0, 0, 0))
ports_4 = [{"position": (0.0, 0.0, 0.15), "direction": (0, 0, -1), "radius": 0.001}]
solid_4, void_4, shell_4, embed_report_4 = run_embedding(domain_4, hier_mesh, ports=ports_4, label="Exp4 embedding")

### 4h. View & Export Embedded STL

In [ ]:
show_mesh_3d(solid_4, title="Exp4 -- Hierarchical Tissue Embedded Solid", color="plum")
export_stl(solid_4, "exp4_hier_solid.stl", "Exp4 solid")